### **Ingestion del archivo "device.csv"**

##### Paso 1 - Leer el archivo CSV usando "DataframeReader" de Spark

In [16]:
%run configuration

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 18, Finished, Available, Finished, True)

In [17]:
%run commom_functions

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 19, Finished, Available, Finished, True)

In [18]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 20, Finished, Available, Finished, True)

In [19]:
device_schema = StructType( fields = [
    StructField("DeviceID", IntegerType(), False),
    StructField("BrandID", IntegerType(), True),
    StructField("ModelID", IntegerType(), True),
    StructField("DisplayID", IntegerType(), True),
    StructField("CameraID", IntegerType(), True),
    StructField("ConnectivityID", IntegerType(), True),
    StructField("OSID", IntegerType(), True),
    StructField("PhysicalSpecID", IntegerType(), True),
    StructField("RleasedYear", IntegerType(), True),
    StructField("ReleasedAnnounced", DateType(), True)
] )


StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 21, Finished, Available, Finished, True)

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
device_df = spark.read \
    .option("header",True) \
    .schema(device_schema) \
    .csv(f"{bronze_folder_path}/{p_file_date}/device.csv")


### **Paso 2 - Seleccionar solo las columnas "requeridas"**

In [21]:
from pyspark.sql.functions import col

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
device_selected_df = device_df.select(col("DeviceID"), col("BrandID"), col("ModelID"), col("DisplayID"), col("CameraID"), col("OSID"),col("PhysicalSpecID"), col("ReleasedAnnounced"))

### **Paso 3 - Cambiar el nombre de las columnas según lo "requerido"**

In [9]:
device_renamed_df = device_selected_df \
    .withColumnRenamed("DeviceID", "device_id") \
    .withColumnRenamed("BrandID", "brand_id") \
    .withColumnRenamed("ModelID", "model_id") \
    .withColumnRenamed("DisplayID", "display_id") \
    .withColumnRenamed("CameraID", "camera_id") \
    .withColumnRenamed("ConnectivityID", "connectivity_id") \
    .withColumnRenamed("OSID", "os_id") \
    .withColumnRenamed("PhysicalSpecID", "physical_spec_id") \
    .withColumnRenamed("ReleasedAnnounced", "released_announced")


StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 11, Finished, Available, Finished, False)

### **Paso 4 - Agregar las columnas "ingestion_date" y "environment"**

In [ ]:
from pyspark.sql.functions import lit

In [11]:
device_final_df = add_ingestion_date(device_renamed_df,p_file_date).withColumn("envrionment",lit(p_environment))

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 13, Finished, Available, Finished, False)

### **Paso 5 - Escribir datos en el "lakehouse" en formato "Delta"**

In [12]:
%%sql
drop table if EXISTS device;

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
device_final_df.write.format("delta").mode("overwrite").saveAsTable("device")

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 15, Finished, Available, Finished, False)

In [14]:
%%sql
select * from device limit 10;

StatementMeta(, ac9a1483-be7a-4a47-8b31-e5b20db8df28, 16, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 11 fields>